# 11 - Five-epoch LoRA domain adaptation

When the classifier is applied to a new domain, the released system performs
five additional epochs of LoRA fine-tuning on a 60% split of the target data
and evaluates on the remaining 40%. This notebook reproduces the adapted Hotel
Reviews and Cross-Domain scenarios.

Adapted adapters are written to `models/adapters/<dataset>/` and are not
committed to Git.

> **Reproduction note.** The original experiments reuse the same 40% split as
> both the adaptation validation set and the evaluation set, so the adapted
> scores are optimistic. That protocol is preserved here to stay comparable
> with the published numbers. Queries are also encoded with
> `sentence-transformers/all-distilroberta-v1` although the FAISS index was
> built with `roberta-base`; see `src/spam_detection/embeddings.py`.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from spam_detection.classifier import (
    fine_tune_adapter,
    load_roberta_model,
    predict_roberta_proba,
)
from spam_detection.config import (
    CROSS_DOMAIN_PATH,
    HOTEL_REVIEWS_PATH,
    K_NEIGHBORS,
    MODELS_DIR,
    RESULTS_DIR,
    TAU_HIGH,
    TAU_LOW,
    VECTOR_DB_INDEX,
    VECTOR_DB_PARQUET,
)
from spam_detection.embeddings import encode_texts
from spam_detection.evaluation import comparison_table, gray_zone_analysis, predictions_from_results, save_table
from spam_detection.pipeline import run_ablation_study
from spam_detection.preprocessing import clean_text_basic, remove_empty_reviews
from spam_detection.prompts import create_hybrid_chain, create_light_chain
from spam_detection.retrieval import load_labeled_faiss_db, precompute_faiss_summaries
from spam_detection.utils import as_int_label, set_seed

set_seed(42)
RUN_LLM = True
RUN_DATASETS = ["hotel_reviews", "cross_domain"]

## Adaptation routine

In [ ]:
def load_external(name: str) -> pd.DataFrame:
    path = HOTEL_REVIEWS_PATH if name == "hotel_reviews" else CROSS_DOMAIN_PATH
    frame = pd.read_csv(path)
    frame = frame.reset_index(drop=True)
    frame["text"] = frame["text"].apply(clean_text_basic)
    frame = remove_empty_reviews(frame, "text", min_length=3, verbose=False)
    frame["label"] = frame["label"].map(as_int_label).astype(int)
    return frame


def adapt_and_evaluate(name: str, frame: pd.DataFrame) -> dict:
    out_dir = RESULTS_DIR / "runs" / f"{name}_adapted"
    out_dir.mkdir(parents=True, exist_ok=True)

    train_df, test_df = train_test_split(frame, test_size=0.4, random_state=42)

    model, tokenizer = load_roberta_model(trainable=True)
    trainer, model, tokenizer = fine_tune_adapter(
        train_df,
        test_df,
        model=model,
        tokenizer=tokenizer,
        output_dir=str(out_dir / "training_run"),
        epochs=5,
        learning_rate=1e-4,
        batch_size=16,
        seed=42,
    )
    model.eval()

    adapter_dir = MODELS_DIR / "adapters" / name
    trainer.save_model(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))

    clf_probas = predict_roberta_proba(test_df["text"].tolist(), model, tokenizer)
    embeddings = encode_texts(test_df["text"].tolist(), batch_size=64, normalize=True)

    faiss_db = load_labeled_faiss_db(VECTOR_DB_PARQUET, VECTOR_DB_INDEX)
    summaries = precompute_faiss_summaries(embeddings, faiss_db, k_neighbors=K_NEIGHBORS)

    clf_only = run_ablation_study(
        df=test_df, clf_probas=clf_probas, true_labels=test_df["label"],
        use_llm=False, use_retrieval=False, use_interpolation=False, show_progress=False,
    )
    save_table(clf_only, out_dir / "clf_only.csv")

    results = {"clf_only": clf_only}
    if RUN_LLM:
        full_rag = run_ablation_study(
            df=test_df, clf_probas=clf_probas, true_labels=test_df["label"],
            llm_chain=create_hybrid_chain(), faiss_summaries=summaries,
            lower_thresh=TAU_LOW, upper_thresh=TAU_HIGH,
            use_llm=True, use_retrieval=True, use_interpolation=False,
        )
        save_table(full_rag, out_dir / "full_rag.csv")
        results["full_rag"] = full_rag

        light_rag = run_ablation_study(
            df=test_df, clf_probas=clf_probas, true_labels=test_df["label"],
            llm_chain=create_light_chain(), faiss_summaries=summaries,
            lower_thresh=TAU_LOW, upper_thresh=TAU_HIGH,
            use_llm=True, use_retrieval=True, use_interpolation=False,
        )
        save_table(light_rag, out_dir / "full_rag_light.csv")
        results["light_rag"] = light_rag

    predictions = {"RoBERTa-LoRA (adapted)": predictions_from_results(clf_only)}
    if "full_rag" in results:
        predictions["Full RAG (adapted)"] = predictions_from_results(results["full_rag"])
        predictions["Full RAG (light prompt)"] = predictions_from_results(results["light_rag"])

    table = comparison_table(test_df["label"], predictions, dataset_name=f"{name} (5-epoch adaptation)")
    display(table)
    save_table(table, RESULTS_DIR / "tables" / f"{name}_adapted_ablation.csv")
    return results

## Run adaptation

In [ ]:
for dataset_name in RUN_DATASETS:
    print(f"=== {dataset_name} ===")
    dataset = load_external(dataset_name)
    print(dataset.shape, dataset["label"].value_counts(normalize=True).round(4).to_dict())
    adapt_and_evaluate(dataset_name, dataset)